<h1>Notebook to explore GROQ APIs and CodeAct prompting techniques</h1>

In [19]:
import json
import os
import re
import requests
from dotenv import load_dotenv
from typing import List, Dict

load_dotenv()

True

In [5]:
def llm_request():
    """
    Show a raw Groq API request and response.
    """
    
    # Your API key
    api_key = os.getenv("GROQ_API_KEY")
    
    # The URL endpoint
    url = "https://api.groq.com/openai/v1/chat/completions"
    
    # HTTP Headers
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    
    # Request body (JSON payload)
    payload = {
        "model": os.getenv("MODEL"),
        "messages": [
            {
                "role": "system",
                "content": "You are a helpful assistant. Respond in JSON format with {\"response\": \"...\"}."
            },
            {
                "role": "user",
                "content": "What is 2+2?"
            }
        ],
        "temperature": 0.7,
        "max_tokens": 1024,
    }
    
    print("=" * 70)
    print("RAW GROQ API REQUEST")
    print("=" * 70)
    print(f"\nMethod: POST")
    print(f"URL: {url}")
    print(f"\nHeaders:")
    for k, v in headers.items():
        if "Bearer" in str(v):
            print(f"  {k}: Bearer <YOUR_API_KEY>")
        else:
            print(f"  {k}: {v}")
    print(f"\nBody (JSON):")
    print(json.dumps(payload, indent=2))
    
    print("\n" + "=" * 70)
    print("SENDING REQUEST...")
    print("=" * 70)
    
    # Make the actual request
    try:
        response = requests.post(url, json=payload, headers=headers, timeout=30)
        response.raise_for_status()
    except Exception as e:
        print(f"\n Request failed: {e}")
        return

    
    
    print(f"\nStatus Code: {response.status_code}")
    print(f"Content-Type: {response.headers.get('content-type')}")
    
    # Parse response
    data = response.json()
    message = data["choices"][0]["message"]["content"]
    
    print("\n" + "=" * 70)
    print("RAW GROQ API RESPONSE")
    print("=" * 70)
    print(json.dumps(data, indent=2))
    
    # Extract the actual message
    print("\n" + "=" * 70)
    print("EXTRACTED MESSAGE")
    print("=" * 70)
    
    print(f"\n{message}")
    
    # Token usage info
    print("\n" + "=" * 70)
    print("TOKEN USAGE")
    print("=" * 70)
    usage = data.get("usage", {})
    print(f"  Input tokens: {usage.get('prompt_tokens')}")
    print(f"  Output tokens: {usage.get('completion_tokens')}")
    print(f"  Total tokens: {usage.get('total_tokens')}")

In [6]:
llm_request()

RAW GROQ API REQUEST

Method: POST
URL: https://api.groq.com/openai/v1/chat/completions

Headers:
  Authorization: Bearer <YOUR_API_KEY>
  Content-Type: application/json

Body (JSON):
{
  "model": "llama-3.3-70b-versatile",
  "messages": [
    {
      "role": "system",
      "content": "You are a helpful assistant. Respond in JSON format with {\"response\": \"...\"}."
    },
    {
      "role": "user",
      "content": "What is 2+2?"
    }
  ],
  "temperature": 0.7,
  "max_tokens": 1024
}

SENDING REQUEST...

Status Code: 200
Content-Type: application/json

RAW GROQ API RESPONSE
{
  "id": "chatcmpl-cb3ec445-ab98-42ae-8740-125657cc286f",
  "object": "chat.completion",
  "created": 1778157604,
  "model": "llama-3.3-70b-versatile",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "{\"response\": \"2 + 2 = 4\"}"
      },
      "logprobs": null,
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "queue_time": 0.062090674,
    "

In [8]:
def groq_codeact():
    """
    Example: CodeAct agent system prompt sent to Groq.
    """
    
    api_key = os.getenv("GROQ_API_KEY")
    url = "https://api.groq.com/openai/v1/chat/completions"
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    
    # CodeAct system prompt
    system_prompt = """You are a CodeAct agent. Solve problems by writing Python code.
 
Format your response:
<THOUGHT>
Explain your approach.
</THOUGHT>
 
<CODE>
# Python code here
result = ...
</CODE>
 
After code execution, if you have the answer:
<FINAL_ANSWER>
Your answer.
</FINAL_ANSWER>
 
Available functions: range, len, sum, max, min, abs, sorted, list, dict, int, float, str, print"""
    
    payload = {
        "model": os.getenv("MODEL"),
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": "What is 5 factorial?"
            }
        ],
        "temperature": 0.7,
        "max_tokens": 1024,
    }
    
    print("=" * 70)
    print("CODEACT EXAMPLE REQUEST")
    print("=" * 70)
    print(f"\nSystem Prompt:\n{system_prompt}\n")
    print(f"User Query: What is 5 factorial?\n")
    
    print("=" * 70)
    print("SENDING REQUEST TO GROQ...")
    print("=" * 70)
    
    try:
        response = requests.post(url, json=payload, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        message = data["choices"][0]["message"]["content"]
        print(f"\nAgent Response:\n{message}")
        
    except Exception as e:
        print(f"Error: {e}")

In [9]:
groq_codeact()

CODEACT EXAMPLE REQUEST

System Prompt:
You are a CodeAct agent. Solve problems by writing Python code.
 
Format your response:
<THOUGHT>
Explain your approach.
</THOUGHT>
 
<CODE>
# Python code here
result = ...
</CODE>
 
After code execution, if you have the answer:
<FINAL_ANSWER>
Your answer.
</FINAL_ANSWER>
 
Available functions: range, len, sum, max, min, abs, sorted, list, dict, int, float, str, print

User Query: What is 5 factorial?

SENDING REQUEST TO GROQ...

Agent Response:
<THOUGHT>
To calculate the factorial of a number, we need to multiply all positive integers less than or equal to that number. In this case, we want to find 5 factorial, which is denoted as 5!. The formula for factorial is n! = n * (n-1) * (n-2) * ... * 2 * 1. We can use a loop in Python to calculate this.
</THOUGHT>

<CODE>
def factorial(n):
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

result = factorial(5)
print(result)
</CODE>

<FINAL_ANSWER>
120
</FINAL_ANSWER>


In [14]:
def llm_request(messages: List[Dict], api_key=os.getenv("GROQ_API_KEY"), model=os.getenv("MODEL"), temperature=0.5, max_tokens= 1024):
    """
    Groq API Request
    Message Structure:
    [
        {
            "role": "system",
            "content": "You are a helpful assistant. Respond in JSON format with {\"response\": \"...\"}."
        },
        {
            "role": "user",
            "content": "What is 2+2?"
        }
    ]
    
    """
    
    
    # The URL endpoint
    url = "https://api.groq.com/openai/v1/chat/completions"
    
    # HTTP Headers
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    
    # Request body (JSON payload)
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    
    # Make the actual request
    try:
        response = requests.post(url, json=payload, headers=headers, timeout=30)
        response.raise_for_status()
    except Exception as e:
        print(f"\n Request failed: {e}")
        return
        
    
    print(f"\nStatus Code: {response.status_code}")
    print(f"Content-Type: {response.headers.get('content-type')}")
    
    # Parse response
    data = response.json()
    message = data["choices"][0]["message"]["content"]
    return message, response.status_code
    

In [15]:
messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Respond in JSON format with {\"response\": \"...\"}."
        },
        {
            "role": "user",
            "content": "What is 2+2?"
        }
    ]

In [17]:
response, status_code = llm_request(
    messages=messages,
)


Status Code: 200
Content-Type: application/json


In [18]:
response

'{"response": "2 + 2 = 4"}'

In [21]:
def tag_content(text, tags):
    """
    This Function extracts the content between HTML tags.
    Input Args:
    text: raw text containing html tags
    tags: list of tags for content extraction

    Output Args:
    res: Dictionary of tags and extracted content. If content found, it's returned with the content else with None for each tag
    """
    res = {}
    for tag in tags:
        pattern = rf"<{tag}>(.*?)</{tag}>"
        match = re.search(pattern, text, re.DOTALL)
        if match:
            res[tag]= match.group(1).strip()
        else:
            res[tag]= None
    return res

In [30]:
def llm_codeact(messages: List[Dict], tags: List):
    """
    CodeAct agent system prompt sent to LLM.
    Example System prompt for CodeACT Agent.
        You are a CodeAct agent. Solve problems by writing Python code.
     
        Format your response:
        <THOUGHT>
        Explain your approach.
        </THOUGHT>
         
        <CODE>
        # Python code here
        result = ...
        </CODE>
         
        After code execution, if you have the answer:
        <FINAL_ANSWER>
        Your answer.
        </FINAL_ANSWER>
         
        Available functions: range, len, sum, max, min, abs, sorted, list, dict, int, float, str, print
    """
    
    response, status = llm_request(
    messages=messages,
        temperature=0.7,
        max_tokens=1024)
    return tag_content(response, tags), status

In [31]:
system_prompt = """You are a CodeAct agent. Solve problems by writing Python code.
 
Format your response:
<THOUGHT>
Explain your approach.
</THOUGHT>
 
<CODE>
# Python code here
result = ...
</CODE>
 
After code execution, if you have the answer:
<FINAL_ANSWER>
Your answer.
</FINAL_ANSWER>
 
Available functions: range, len, sum, max, min, abs, sorted, list, dict, int, float, str, print"""

messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": "What is 5 factorial?"
            }
        ]
tags = ["THOUGHT", "CODE", "FINAL_ANSWER"]

In [33]:
response, status_code = llm_codeact(
    messages=messages,
    tags=tags
)


Status Code: 200
Content-Type: application/json


In [35]:
response

{'THOUGHT': 'To find the factorial of a number, we need to multiply all the positive integers less than or equal to that number. In this case, we want to calculate 5 factorial, which is the product of all positive integers from 1 to 5. We can use a loop to calculate this product.',
 'CODE': 'result = 1\nfor i in range(1, 6):\n    result *= i\nprint(result)',
 'FINAL_ANSWER': '120'}